# Exploratory Data Analysis - Plant Disease Detection

**Authors:** Pratham Rajesh, Shreram Palanisamy  
**Date:** November 2025  
**Dataset:** PlantVillage

This notebook performs comprehensive exploratory data analysis on the PlantVillage dataset to understand:
- Dataset structure and organization
- Class distribution and imbalance
- Image characteristics (dimensions, formats, quality)
- Sample visualizations (healthy vs diseased leaves)
- Statistical properties for normalization
- Train/validation/test split creation

## Table of Contents
1. [Setup & Imports](#setup)
2. [Dataset Loading](#loading)
3. [Class Distribution Analysis](#distribution)
4. [Sample Visualization](#visualization)
5. [Image Statistics](#statistics)
6. [Train/Val/Test Split](#split)
7. [Summary & Insights](#summary)

<a id='setup'></a>
## 1. Setup & Imports

In [ ]:
# Standard libraries
import os
import json
from pathlib import Path
from collections import Counter

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# Machine Learning
from sklearn.model_selection import train_test_split

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

# Set random seed for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print("Setup complete!")

<a id='loading'></a>
## 2. Dataset Loading

In [ ]:
# Define dataset path
DATASET_PATH = Path('../Plant_leave_diseases_dataset_without_augmentation')

# Verify dataset exists
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {DATASET_PATH}")

print(f"Dataset path: {DATASET_PATH}")
print(f"Dataset exists: {DATASET_PATH.exists()}")

In [ ]:
# Load dataset structure
def load_dataset_info(dataset_path):
    """
    Load dataset information including class names and file paths.
    
    Returns:
        dict: Dictionary containing class names, file paths, and counts
    """
    data = {
        'class_names': [],
        'file_paths': [],
        'labels': [],
        'class_counts': {}
    }
    
    # Get all class directories
    class_dirs = sorted([d for d in dataset_path.iterdir() if d.is_dir()])
    
    for class_idx, class_dir in enumerate(class_dirs):
        class_name = class_dir.name
        data['class_names'].append(class_name)
        
        # Get all image files in this class
        image_files = list(class_dir.glob('*.JPG')) + list(class_dir.glob('*.jpg')) + \
                     list(class_dir.glob('*.PNG')) + list(class_dir.glob('*.png'))
        
        data['class_counts'][class_name] = len(image_files)
        
        # Add to file paths and labels
        for img_path in image_files:
            data['file_paths'].append(str(img_path))
            data['labels'].append(class_idx)
    
    return data

# Load dataset
print("Loading dataset information...")
dataset_info = load_dataset_info(DATASET_PATH)

print(f"\nDataset Summary:")
print(f"  Total classes: {len(dataset_info['class_names'])}")
print(f"  Total images: {len(dataset_info['file_paths'])}")
print(f"\nFirst 5 classes:")
for i, class_name in enumerate(dataset_info['class_names'][:5]):
    print(f"  {i}: {class_name} ({dataset_info['class_counts'][class_name]} images)")

<a id='distribution'></a>
## 3. Class Distribution Analysis

In [ ]:
# Create DataFrame for analysis
class_df = pd.DataFrame([
    {'Class': class_name, 'Count': count}
    for class_name, count in dataset_info['class_counts'].items()
]).sort_values('Count', ascending=False).reset_index(drop=True)

print("Class Distribution Statistics:")
print(f"  Mean samples per class: {class_df['Count'].mean():.0f}")
print(f"  Median samples per class: {class_df['Count'].median():.0f}")
print(f"  Min samples: {class_df['Count'].min()} ({class_df.loc[class_df['Count'].idxmin(), 'Class']})")
print(f"  Max samples: {class_df['Count'].max()} ({class_df.loc[class_df['Count'].idxmax(), 'Class']})")
print(f"  Std deviation: {class_df['Count'].std():.0f}")
print(f"  Imbalance ratio: {class_df['Count'].max() / class_df['Count'].min():.2f}x")

# Display top 10 and bottom 10 classes
print("\nTop 10 classes (most samples):")
print(class_df.head(10).to_string(index=False))

print("\nBottom 10 classes (least samples):")
print(class_df.tail(10).to_string(index=False))

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(2, 1, figsize=(16, 12))

# Plot 1: Full distribution bar chart
ax1 = axes[0]
bars = ax1.bar(range(len(class_df)), class_df['Count'], color='steelblue', edgecolor='black', alpha=0.7)

# Color code bars by sample count
colors = ['red' if c < 500 else 'orange' if c < 1000 else 'yellow' if c < 2000 else 'green' 
          for c in class_df['Count']]
for bar, color in zip(bars, colors):
    bar.set_color(color)
    bar.set_alpha(0.7)

ax1.set_xlabel('Class Index', fontsize=12, fontweight='bold')
ax1.set_ylabel('Number of Samples', fontsize=12, fontweight='bold')
ax1.set_title('Class Distribution - All 39 Classes', fontsize=14, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Add mean line
mean_count = class_df['Count'].mean()
ax1.axhline(y=mean_count, color='black', linestyle='--', linewidth=2, label=f'Mean: {mean_count:.0f}')
ax1.legend(fontsize=10)

# Legend for colors
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='red', alpha=0.7, label='< 500 samples'),
    Patch(facecolor='orange', alpha=0.7, label='500-1000 samples'),
    Patch(facecolor='yellow', alpha=0.7, label='1000-2000 samples'),
    Patch(facecolor='green', alpha=0.7, label='> 2000 samples')
]
ax1.legend(handles=legend_elements, loc='upper right', fontsize=9)

# Plot 2: Horizontal bar chart for top/bottom classes
ax2 = axes[1]
top_bottom = pd.concat([class_df.head(10), class_df.tail(10)])
y_pos = range(len(top_bottom))
bars2 = ax2.barh(y_pos, top_bottom['Count'], color='teal', edgecolor='black', alpha=0.7)

# Shorten class names for display
short_names = [name[:40] + '...' if len(name) > 40 else name for name in top_bottom['Class']]
ax2.set_yticks(y_pos)
ax2.set_yticklabels(short_names, fontsize=9)
ax2.set_xlabel('Number of Samples', fontsize=12, fontweight='bold')
ax2.set_title('Top 10 and Bottom 10 Classes by Sample Count', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

# Add value labels
for i, v in enumerate(top_bottom['Count']):
    ax2.text(v + 50, i, str(v), va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("Class distribution plot saved to: reports/figures/class_distribution.png")

In [ ]:
# Analyze plant types
plant_types = {}
for class_name in dataset_info['class_names']:
    if '___' in class_name:
        plant = class_name.split('___')[0]
    else:
        plant = class_name
    
    if plant not in plant_types:
        plant_types[plant] = 0
    plant_types[plant] += dataset_info['class_counts'][class_name]

plant_df = pd.DataFrame([
    {'Plant': plant, 'Total_Images': count}
    for plant, count in plant_types.items()
]).sort_values('Total_Images', ascending=False)

print("\nSamples per plant type:")
print(plant_df.to_string(index=False))

In [ ]:
# Visualize plant type distribution
fig, ax = plt.subplots(figsize=(12, 8))

bars = ax.bar(range(len(plant_df)), plant_df['Total_Images'], 
              color='forestgreen', edgecolor='black', alpha=0.7)
ax.set_xticks(range(len(plant_df)))
ax.set_xticklabels(plant_df['Plant'], rotation=45, ha='right', fontsize=10)
ax.set_xlabel('Plant Type', fontsize=12, fontweight='bold')
ax.set_ylabel('Total Number of Images', fontsize=12, fontweight='bold')
ax.set_title('Distribution of Images Across Plant Types', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

# Add value labels
for i, v in enumerate(plant_df['Total_Images']):
    ax.text(i, v + 100, str(v), ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/plant_type_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("Plant type distribution plot saved to: reports/figures/plant_type_distribution.png")

<a id='visualization'></a>
## 4. Sample Visualization

In [ ]:
# Display sample images - Healthy vs Diseased
def display_samples(dataset_info, num_samples=16):
    """
    Display grid of sample images showing healthy and diseased leaves.
    """
    # Get healthy and diseased classes
    healthy_classes = [c for c in dataset_info['class_names'] if 'healthy' in c.lower()]
    diseased_classes = [c for c in dataset_info['class_names'] if 'healthy' not in c.lower() 
                       and 'background' not in c.lower()]
    
    # Select subset for visualization
    selected_healthy = np.random.choice(healthy_classes, min(4, len(healthy_classes)), replace=False)
    selected_diseased = np.random.choice(diseased_classes, min(12, len(diseased_classes)), replace=False)
    
    fig, axes = plt.subplots(4, 4, figsize=(16, 16))
    axes = axes.ravel()
    
    idx = 0
    
    # Plot healthy samples
    for class_name in selected_healthy:
        class_path = DATASET_PATH / class_name
        images = list(class_path.glob('*.JPG')) + list(class_path.glob('*.jpg'))
        if images:
            img_path = np.random.choice(images)
            img = Image.open(img_path)
            
            axes[idx].imshow(img)
            axes[idx].set_title(f"HEALTHY\n{class_name.split('___')[0]}", 
                               fontsize=10, fontweight='bold', color='green')
            axes[idx].axis('off')
            idx += 1
    
    # Plot diseased samples
    for class_name in selected_diseased:
        class_path = DATASET_PATH / class_name
        images = list(class_path.glob('*.JPG')) + list(class_path.glob('*.jpg'))
        if images:
            img_path = np.random.choice(images)
            img = Image.open(img_path)
            
            # Parse class name
            if '___' in class_name:
                plant, disease = class_name.split('___')
                disease = disease.replace('_', ' ')[:30]
                title = f"DISEASED\n{plant}: {disease}"
            else:
                title = class_name[:40]
            
            axes[idx].imshow(img)
            axes[idx].set_title(title, fontsize=9, fontweight='bold', color='red')
            axes[idx].axis('off')
            idx += 1
    
    plt.suptitle('Sample Images: Healthy vs Diseased Plant Leaves', 
                fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig('../reports/figures/sample_images.png', dpi=300, bbox_inches='tight')
    plt.show()

display_samples(dataset_info)
print("Sample images saved to: reports/figures/sample_images.png")

<a id='statistics'></a>
## 5. Image Statistics

In [ ]:
# Analyze image properties
def analyze_image_properties(file_paths, sample_size=1000):
    """
    Analyze image dimensions, formats, and pixel statistics.
    """
    # Sample random images
    sampled_paths = np.random.choice(file_paths, min(sample_size, len(file_paths)), replace=False)
    
    dimensions = []
    formats = []
    pixel_values = []
    
    print(f"Analyzing {len(sampled_paths)} sample images...")
    
    for img_path in sampled_paths:
        try:
            img = Image.open(img_path)
            dimensions.append(img.size)  # (width, height)
            formats.append(img.format)
            
            # Convert to numpy for pixel statistics
            img_array = np.array(img) / 255.0  # Normalize to [0, 1]
            pixel_values.append(img_array)
            
        except Exception as e:
            print(f"Error processing {img_path}: {e}")
    
    # Compute statistics
    unique_dimensions = Counter(dimensions)
    unique_formats = Counter(formats)
    
    # Pixel statistics (for normalization)
    all_pixels = np.concatenate([img.reshape(-1, 3) for img in pixel_values], axis=0)
    mean_rgb = all_pixels.mean(axis=0)
    std_rgb = all_pixels.std(axis=0)
    
    return {
        'dimensions': unique_dimensions,
        'formats': unique_formats,
        'mean_rgb': mean_rgb,
        'std_rgb': std_rgb
    }

# Run analysis
image_stats = analyze_image_properties(dataset_info['file_paths'], sample_size=1000)

print("\n" + "="*60)
print("IMAGE STATISTICS")
print("="*60)

print("\nImage Dimensions:")
for dim, count in image_stats['dimensions'].most_common():
    print(f"  {dim[0]}x{dim[1]}: {count} images")

print("\nImage Formats:")
for fmt, count in image_stats['formats'].most_common():
    print(f"  {fmt}: {count} images")

print("\nPixel Value Statistics (normalized to [0, 1]):")
print(f"  Mean RGB: [{image_stats['mean_rgb'][0]:.4f}, {image_stats['mean_rgb'][1]:.4f}, {image_stats['mean_rgb'][2]:.4f}]")
print(f"  Std RGB:  [{image_stats['std_rgb'][0]:.4f}, {image_stats['std_rgb'][1]:.4f}, {image_stats['std_rgb'][2]:.4f}]")

print("\nImageNet Statistics (for transfer learning):")
print("  Mean RGB: [0.485, 0.456, 0.406]")
print("  Std RGB:  [0.229, 0.224, 0.225]")
print("\nRecommendation: Use ImageNet statistics for transfer learning with ResNet50.")

<a id='split'></a>
## 6. Train/Validation/Test Split

In [ ]:
# Create stratified train/val/test split (70/15/15)
def create_stratified_split(file_paths, labels, train_size=0.7, val_size=0.15, test_size=0.15, random_state=42):
    """
    Create stratified train/validation/test split.
    
    Args:
        file_paths: List of file paths
        labels: List of labels (class indices)
        train_size: Proportion for training (default 0.7)
        val_size: Proportion for validation (default 0.15)
        test_size: Proportion for testing (default 0.15)
        random_state: Random seed for reproducibility
    
    Returns:
        dict: Dictionary containing train/val/test indices and paths
    """
    assert abs(train_size + val_size + test_size - 1.0) < 1e-6, "Splits must sum to 1.0"
    
    # First split: train vs (val + test)
    train_paths, temp_paths, train_labels, temp_labels = train_test_split(
        file_paths, labels, 
        train_size=train_size, 
        stratify=labels,
        random_state=random_state
    )
    
    # Second split: val vs test
    val_ratio = val_size / (val_size + test_size)
    val_paths, test_paths, val_labels, test_labels = train_test_split(
        temp_paths, temp_labels,
        train_size=val_ratio,
        stratify=temp_labels,
        random_state=random_state
    )
    
    return {
        'train': {'paths': train_paths, 'labels': train_labels},
        'val': {'paths': val_paths, 'labels': val_labels},
        'test': {'paths': test_paths, 'labels': test_labels}
    }

# Create split
print("Creating stratified train/val/test split (70/15/15)...")
splits = create_stratified_split(
    dataset_info['file_paths'], 
    dataset_info['labels'],
    train_size=0.7,
    val_size=0.15,
    test_size=0.15,
    random_state=RANDOM_SEED
)

print("\nSplit Statistics:")
print(f"  Training set:   {len(splits['train']['paths']):,} images ({len(splits['train']['paths'])/len(dataset_info['file_paths'])*100:.1f}%)")
print(f"  Validation set: {len(splits['val']['paths']):,} images ({len(splits['val']['paths'])/len(dataset_info['file_paths'])*100:.1f}%)")
print(f"  Test set:       {len(splits['test']['paths']):,} images ({len(splits['test']['paths'])/len(dataset_info['file_paths'])*100:.1f}%)")
print(f"  Total:          {len(dataset_info['file_paths']):,} images")

In [ ]:
# Verify stratification - check class distribution in each split
train_class_counts = Counter(splits['train']['labels'])
val_class_counts = Counter(splits['val']['labels'])
test_class_counts = Counter(splits['test']['labels'])

# Create verification DataFrame
split_verification = []
for class_idx, class_name in enumerate(dataset_info['class_names']):
    total = dataset_info['class_counts'][class_name]
    train_count = train_class_counts.get(class_idx, 0)
    val_count = val_class_counts.get(class_idx, 0)
    test_count = test_class_counts.get(class_idx, 0)
    
    split_verification.append({
        'Class': class_name[:40],
        'Total': total,
        'Train': train_count,
        'Val': val_count,
        'Test': test_count,
        'Train%': f"{train_count/total*100:.1f}",
        'Val%': f"{val_count/total*100:.1f}",
        'Test%': f"{test_count/total*100:.1f}"
    })

split_df = pd.DataFrame(split_verification)

print("\nSample of stratification verification (first 10 classes):")
print(split_df.head(10).to_string(index=False))

print("\nStratification Check:")
print(f"  All classes present in train: {all(train_class_counts.get(i, 0) > 0 for i in range(len(dataset_info['class_names'])))}")
print(f"  All classes present in val:   {all(val_class_counts.get(i, 0) > 0 for i in range(len(dataset_info['class_names'])))}")
print(f"  All classes present in test:  {all(test_class_counts.get(i, 0) > 0 for i in range(len(dataset_info['class_names'])))}")

In [ ]:
# Save split information for reproducibility
split_info = {
    'class_names': dataset_info['class_names'],
    'random_seed': RANDOM_SEED,
    'train_paths': splits['train']['paths'],
    'train_labels': splits['train']['labels'],
    'val_paths': splits['val']['paths'],
    'val_labels': splits['val']['labels'],
    'test_paths': splits['test']['paths'],
    'test_labels': splits['test']['labels']
}

# Save to JSON
output_path = Path('../models/dataset_split.json')
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, 'w') as f:
    json.dump(split_info, f, indent=2)

print(f"\nSplit information saved to: {output_path}")

# Also save class names separately
class_names_path = Path('../models/class_names.json')
with open(class_names_path, 'w') as f:
    json.dump(dataset_info['class_names'], f, indent=2)

print(f"Class names saved to: {class_names_path}")

<a id='summary'></a>
## 7. Summary & Insights

In [ ]:
# Generate comprehensive summary
print("="*80)
print("EXPLORATORY DATA ANALYSIS - SUMMARY")
print("="*80)

print("\n1. DATASET OVERVIEW")
print(f"   - Total images: {len(dataset_info['file_paths']):,}")
print(f"   - Number of classes: {len(dataset_info['class_names'])}")
print(f"   - Image format: JPEG (256x256 RGB)")
print(f"   - Plant types covered: {len(plant_types)}")

print("\n2. CLASS DISTRIBUTION")
print(f"   - Mean samples per class: {class_df['Count'].mean():.0f}")
print(f"   - Median samples per class: {class_df['Count'].median():.0f}")
print(f"   - Min samples: {class_df['Count'].min()} ({class_df.loc[class_df['Count'].idxmin(), 'Class'][:40]})")
print(f"   - Max samples: {class_df['Count'].max()} ({class_df.loc[class_df['Count'].idxmax(), 'Class'][:40]})")
print(f"   - Imbalance ratio: {class_df['Count'].max() / class_df['Count'].min():.2f}x")

print("\n3. IMAGE STATISTICS")
print(f"   - Standard dimension: 256x256 pixels")
print(f"   - Color space: RGB")
print(f"   - Pixel value range: [0, 255]")
print(f"   - Mean RGB (normalized): [{image_stats['mean_rgb'][0]:.3f}, {image_stats['mean_rgb'][1]:.3f}, {image_stats['mean_rgb'][2]:.3f}]")
print(f"   - Std RGB (normalized): [{image_stats['std_rgb'][0]:.3f}, {image_stats['std_rgb'][1]:.3f}, {image_stats['std_rgb'][2]:.3f}]")

print("\n4. DATA SPLITS")
print(f"   - Training set: {len(splits['train']['paths']):,} images (70%)")
print(f"   - Validation set: {len(splits['val']['paths']):,} images (15%)")
print(f"   - Test set: {len(splits['test']['paths']):,} images (15%)")
print(f"   - Stratification: Maintained across all splits")

print("\n5. KEY INSIGHTS")
print("   ✓ Dataset is well-organized with clear class structure")
print("   ✓ Severe class imbalance exists (36x ratio)")
print("   ✓ All images are consistent size (256x256)")
print("   ✓ Multiple plant types with varying disease categories")
print("   ⚠ Need to handle class imbalance (stratified sampling, augmentation)")
print("   ⚠ Smallest classes have <500 samples (may affect performance)")

print("\n6. RECOMMENDATIONS FOR MODELING")
print("   1. Use ImageNet normalization for transfer learning")
print("   2. Apply stratified train/val/test splits (already done)")
print("   3. Use data augmentation to handle class imbalance")
print("   4. Monitor per-class metrics, not just overall accuracy")
print("   5. Consider class weights in loss function")
print("   6. Resize images to 224x224 for ResNet50 input")
print("   7. Use batch size of 32 for stable training")

print("\n7. FILES GENERATED")
print("   - reports/figures/class_distribution.png")
print("   - reports/figures/plant_type_distribution.png")
print("   - reports/figures/sample_images.png")
print("   - models/dataset_split.json")
print("   - models/class_names.json")

print("\n" + "="*80)
print("EDA COMPLETE - Ready for Model Training")
print("="*80)